In [2]:
import geopandas as gpd

gdf = gpd.read_file("hex_clean_sepa.gpkg")   # already has both flood columns

SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"

sepa_flooded = gdf[SEPA_FRAC] > 0
s1_flooded   = gdf[S1_FRAC]   > 0

intersection = sepa_flooded & s1_flooded     # flooded in BOTH
union        = sepa_flooded | s1_flooded
sepa_only    = sepa_flooded & ~s1_flooded
s1_only      = s1_flooded   & ~sepa_flooded

print(f"Total hexes            : {len(gdf):,}")
print(f"SEPA flooded           : {sepa_flooded.sum():,}")
print(f"S1   flooded           : {s1_flooded.sum():,}")
print(f"Intersection (both)    : {intersection.sum():,}")
print(f"Union (either)         : {union.sum():,}")
print(f"SEPA-only (not S1)     : {sepa_only.sum():,}")
print(f"S1-only (not SEPA)     : {s1_only.sum():,}")
if s1_flooded.sum():
    print(f"\n{100*intersection.sum()/s1_flooded.sum():.1f}% of S1-flooded hexes are also SEPA-flooded")
if sepa_flooded.sum():
    print(f"{100*intersection.sum()/sepa_flooded.sum():.1f}% of SEPA-flooded hexes are also S1-flooded")

Total hexes            : 3,276
SEPA flooded           : 3,276
S1   flooded           : 537
Intersection (both)    : 537
Union (either)         : 3,276
SEPA-only (not S1)     : 2,739
S1-only (not SEPA)     : 0

100.0% of S1-flooded hexes are also SEPA-flooded
16.4% of SEPA-flooded hexes are also S1-flooded


In [ ]:
import geopandas as gpd


# This is the pre-cleaning joined grid (before SEPA zeros were dropped).
FULL_PATH = "hex_h3_res9_joined.gpkg"
SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"

full = gpd.read_file(FULL_PATH)

# confirm both flood columns exist
for col in (SEPA_FRAC, S1_FRAC):
    if col not in full.columns:
        raise KeyError(f"{col!r} not in {FULL_PATH}. Columns: {list(full.columns)}")

sepa_flooded = full[SEPA_FRAC] > 0
s1_flooded   = full[S1_FRAC]   > 0

intersection = sepa_flooded & s1_flooded     # flooded in BOTH
union        = sepa_flooded | s1_flooded
sepa_only    = sepa_flooded & ~s1_flooded
s1_only      = s1_flooded   & ~sepa_flooded  # S1 floods where SEPA does NOT

print(f"===== FULL GRID: {FULL_PATH} =====")
print(f"Total hexes            : {len(full):,}")
print(f"SEPA flooded           : {sepa_flooded.sum():,}  ({100*sepa_flooded.mean():.1f}%)")
print(f"S1   flooded           : {s1_flooded.sum():,}  ({100*s1_flooded.mean():.1f}%)")
print(f"Intersection (both)    : {intersection.sum():,}")
print(f"Union (either)         : {union.sum():,}")
print(f"SEPA-only (not S1)     : {sepa_only.sum():,}")
print(f"S1-only (not SEPA)     : {s1_only.sum():,}   <-- key test: is S1 fully nested in SEPA?")
print(f"Neither (dry in both)  : {(~union).sum():,}")

if s1_flooded.sum():
    print(f"\n{100*intersection.sum()/s1_flooded.sum():.1f}% of S1-flooded hexes are also SEPA-flooded")
if sepa_flooded.sum():
    print(f"{100*intersection.sum()/sepa_flooded.sum():.1f}% of SEPA-flooded hexes are also S1-flooded")

# The nesting verdict 
if s1_only.sum() == 0:
    print("\n=> S1 flooding is FULLY NESTED within SEPA: every observed flood "
          "location was SEPA-designated. SEPA missed no observed flooding.")
else:
    print(f"\n=> S1 is NOT fully nested: {s1_only.sum():,} hexes flooded in the storm "
          "but were NOT SEPA-designated (SEPA under-prediction). Worth mapping these.")

===== FULL GRID: hex_h3_res9_joined.gpkg =====
Total hexes            : 4,201
SEPA flooded           : 3,314  (78.9%)
S1   flooded           : 627  (14.9%)
Intersection (both)    : 550
Union (either)         : 3,391
SEPA-only (not S1)     : 2,764
S1-only (not SEPA)     : 77   <-- key test: is S1 fully nested in SEPA?
Neither (dry in both)  : 810

87.7% of S1-flooded hexes are also SEPA-flooded
16.6% of SEPA-flooded hexes are also S1-flooded

=> S1 is NOT fully nested: 77 hexes flooded in the storm but were NOT SEPA-designated (SEPA under-prediction). Worth mapping these.


In [ ]:
import geopandas as gpd
import numpy as np
from libpysal.weights import Queen, KNN
from scipy.spatial import cKDTree

# Load full grid and flag the 550 intersection 
full = gpd.read_file("hex_h3_res9_joined.gpkg")  
if full.crs is None or full.crs.to_epsg() != 3035:
    full = full.to_crs("EPSG:3035")

both = (full["sepa_flood_frac"] > 0) & (full["s1_flood_frac"] > 0)
inter = full[both].reset_index(drop=True)
n = len(inter)
print(f"Intersection hexes: {n}")

# centroids (metric)
cent = inter.geometry.centroid
xy = np.column_stack([cent.x.values, cent.y.values])

# Connected components 
# Queen contiguity on the SUBSET: How many disconnected islands?
w = Queen.from_dataframe(inter, use_index=False)
n_components = w.n_components
comp_labels = w.component_labels
sizes = np.bincount(comp_labels)
print(f"\n[1] Connected components (Queen): {n_components}")
print(f"    largest component: {sizes.max()} hexes ({100*sizes.max()/n:.0f}% of sample)")
print(f"    singletons (isolated hexes): {(sizes==1).sum()}")
print(f"    components with <5 hexes: {(sizes<5).sum()}")
# interpretation: Few big components = compact (good); many singletons = fragmented (bad)

# Isolation: distance to nearest neighbour within the 550 
tree = cKDTree(xy)
d, _ = tree.query(xy, k=2)          # k=2: self + nearest
nn_dist = d[:, 1]
# hex "diameter" for reference (res-9 H3 ~ 174 m edge, ~300 m across)
print(f"\n[2] Nearest-neighbour distance within the 550 (metres):")
print(f"    median {np.median(nn_dist):.0f} | mean {nn_dist.mean():.0f} | "
      f"90th pct {np.percentile(nn_dist,90):.0f} | max {nn_dist.max():.0f}")
# if median NN distance ~ one hex width, they're touching (clustered);
# large 90th/max = some isolated outliers

# Local density: how many of the 550 fall within a GWR-sized window 
# For a candidate bandwidth k, how many neighbours does each hex actually have nearby?
for k in [30, 50, 100]:
    if k < n:
        dk, _ = tree.query(xy, k=k+1)
        radius_k = dk[:, -1]        # distance to k-th neighbour
        print(f"\n[3] For bandwidth={k}: distance to {k}th neighbour (metres)")
        print(f"    median {np.median(radius_k):.0f} | max {radius_k.max():.0f}")
        # large/variable radius = uneven density; local windows span very different areas

# Spatial extent and coverage 
xr = xy[:,0].max() - xy[:,0].min()
yr = xy[:,1].max() - xy[:,1].min()
print(f"\n[4] Bounding-box extent: {xr/1000:.1f} km x {yr/1000:.1f} km")
hull_area = inter.geometry.union_all().convex_hull.area / 1e6
print(f"    convex-hull area: {hull_area:.1f} km^2")
print(f"    hex density in hull: {n/hull_area:.1f} hexes/km^2")

# Check
frag = (sizes==1).sum() / n
big = sizes.max() / n
print("\n===== VERDICT =====")
if big > 0.5 and frag < 0.1:
    print("Compact: majority in one block, few isolates -> MGWR should behave.")
elif frag > 0.3:
    print("Fragmented: many isolated hexes -> local windows will span large areas; "
          "MGWR bandwidths will be forced large/unstable. Floor bandwidth, be cautious.")
else:
    print("Mixed: some clustering with scattered outliers -> MGWR runnable but "
          "watch fine-bandwidth stability; consider a bandwidth floor.")

Intersection hexes: 550

[1] Connected components (Queen): 145
    largest component: 144 hexes (26% of sample)
    singletons (isolated hexes): 75
    components with <5 hexes: 127

[2] Nearest-neighbour distance within the 550 (metres):
    median 317 | mean 383 | 90th pct 576 | max 1563

[3] For bandwidth=30: distance to 30th neighbour (metres)
    median 2151 | max 9754

[3] For bandwidth=50: distance to 50th neighbour (metres)
    median 3293 | max 12768

[3] For bandwidth=100: distance to 100th neighbour (metres)
    median 5017 | max 16302

[4] Bounding-box extent: 30.2 km x 24.9 km
    convex-hull area: 560.4 km^2
    hex density in hull: 1.0 hexes/km^2

===== VERDICT =====
Mixed: some clustering with scattered outliers -> MGWR runnable but watch fine-bandwidth stability; consider a bandwidth floor.


/opt/anaconda3/envs/spatial-regression/lib/python3.14/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 145 disconnected components.
 There are 75 islands with ids: 24, 50, 51, 52, 58, 62, 65, 103, 108, 111, 115, 116, 127, 142, 151, 158, 161, 165, 168, 169, 172, 188, 199, 204, 212, 217, 233, 235, 237, 242, 243, 285, 287, 288, 290, 292, 297, 302, 311, 317, 318, 328, 332, 333, 347, 355, 361, 365, 367, 369, 374, 378, 380, 383, 387, 390, 410, 411, 416, 423, 431, 448, 455, 456, 464, 467, 473, 474, 477, 479, 500, 502, 525, 536, 546.
  W.__init__(self, neighbors, ids=ids, **kw)


# Disagreement-zone analysis: representational (in)justice in SEPA vs Sentinel-1.

Every hex is classified by agreement between observed (S1) and modelled (SEPA)
flooding:
    BOTH     - flooded in both            (SEPA hit)
    S1_ONLY  - flooded in S1, not SEPA    (SEPA MISS / omission)
    SEPA_ONLY- flooded in SEPA, not S1    (SEPA OVER-designation)
    NEITHER  - dry in both                (correct null)

Two questions:
 Q1 (EJ core): are SEPA's MISSES (S1_ONLY) concentrated in more vulnerable areas
               than its HITS (BOTH)? If SEPA overlooks real flooding where
               vulnerable people live, that is representational injustice.
 Q2 (context): do SEPA's OVER-designations (SEPA_ONLY) differ in vulnerability
               from its HITS (BOTH)?

Vulnerability variables are tested as the EJ evidence; hazard variables are tested
as a mechanism check (e.g. are misses far from rivers -> pluvial flooding SEPA's
fluvial model can't see?).

Tests: Mann-Whitney U (non-parametric, no normality assumption) with rank-biserial
effect size and median differences. NOTE the spatial-autocorrelation caveat below.


In [ ]:
import numpy as np
import geopandas as gpd
from scipy.stats import mannwhitneyu


FULL_PATH = "hex_h3_res9_joined.gpkg"
SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"

VULN_VARS = ["simd_rank", "age_dep_ratio", "pop_density"]     # social/vulnerability
HAZARD_VARS = ["dist_to_river_m", "impervious_pct", "mean_slope"]  # mechanism check


SIMD_LOW_IS_DEPRIVED = True    # True if low simd_rank = more deprived

# Load and classify 
gdf = gpd.read_file(FULL_PATH)
sepa_f = gdf[SEPA_FRAC] > 0
s1_f   = gdf[S1_FRAC]   > 0

zone = np.full(len(gdf), "NEITHER", dtype=object)
zone[( s1_f &  sepa_f)] = "BOTH"
zone[( s1_f & ~sepa_f)] = "S1_ONLY"
zone[(~s1_f &  sepa_f)] = "SEPA_ONLY"
gdf["zone"] = zone

counts = {z: int((zone == z).sum()) for z in ["BOTH","S1_ONLY","SEPA_ONLY","NEITHER"]}
print("Zone counts:", counts)
print(f"  BOTH (hits)            : {counts['BOTH']}")
print(f"  S1_ONLY (SEPA misses)  : {counts['S1_ONLY']}")
print(f"  SEPA_ONLY (over-desig) : {counts['SEPA_ONLY']}")
print(f"  NEITHER (correct null) : {counts['NEITHER']}")

def summarise(varname, mask):
    v = gdf.loc[mask, varname].dropna()
    return len(v), v.median(), v.mean(), v.quantile(.25), v.quantile(.75)

def compare(varname, mask_a, mask_b, label_a, label_b):
    a = gdf.loc[mask_a, varname].dropna().to_numpy()
    b = gdf.loc[mask_b, varname].dropna().to_numpy()
    if len(a) < 3 or len(b) < 3:
        return None
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    # rank-biserial effect size: 1 - 2U/(n1 n2)
    rbc = 1 - 2*U/(len(a)*len(b))
    return dict(na=len(a), nb=len(b), med_a=np.median(a), med_b=np.median(b),
               diff=np.median(a)-np.median(b), U=U, p=p, rbc=rbc)

both_m  = gdf["zone"]=="BOTH"
s1_m    = gdf["zone"]=="S1_ONLY"
sepa_m  = gdf["zone"]=="SEPA_ONLY"

#  Q1: SEPA misses vs hits (the EJ test) 
print("\n" + "="*88)
print("Q1  SEPA MISSES (S1_ONLY) vs HITS (BOTH) - are overlooked-flood areas more vulnerable?")
print("="*88)
print(f"{'variable':<20}{'med_miss':>10}{'med_hit':>10}{'diff':>10}{'effect':>9}{'p':>9}")
for v in VULN_VARS + HAZARD_VARS:
    r = compare(v, s1_m, both_m, "miss", "hit")
    if r:
        tag = "*" if r["p"]<0.05 else " "
        print(f"{v:<20}{r['med_a']:>10.3f}{r['med_b']:>10.3f}{r['diff']:>10.3f}"
              f"{r['rbc']:>9.2f}{r['p']:>9.3f}{tag}")

# Q2: SEPA over-designations vs hits 
print("\n" + "="*88)
print("Q2  SEPA OVER-DESIGNATIONS (SEPA_ONLY) vs HITS (BOTH) - do flagged-but-dry areas differ?")
print("="*88)
print(f"{'variable':<20}{'med_over':>10}{'med_hit':>10}{'diff':>10}{'effect':>9}{'p':>9}")
for v in VULN_VARS + HAZARD_VARS:
    r = compare(v, sepa_m, both_m, "over", "hit")
    if r:
        tag = "*" if r["p"]<0.05 else " "
        print(f"{v:<20}{r['med_a']:>10.3f}{r['med_b']:>10.3f}{r['diff']:>10.3f}"
              f"{r['rbc']:>9.2f}{r['p']:>9.3f}{tag}")

# EJ verdict on deprivation for the misses 
print("\n" + "="*88)
print("EJ VERDICT - are SEPA's MISSES concentrated in DEPRIVED areas?")
r = compare("simd_rank", s1_m, both_m, "miss", "hit")
if r:
    if r["p"] < 0.05:
        # translate median difference into deprivation direction
        miss_more_deprived = (r["diff"] < 0) if SIMD_LOW_IS_DEPRIVED else (r["diff"] > 0)
        if miss_more_deprived:
            print("  SEPA's misses fall in MORE deprived areas than its hits "
                  f"(median simd_rank {r['med_a']:.0f} vs {r['med_b']:.0f}, p={r['p']:.3f}).")
            print("  => Evidence of representational injustice: overlooked flooding is "
                  "concentrated on more deprived populations.")
        else:
            print("  SEPA's misses fall in LESS deprived areas than its hits "
                  f"(median {r['med_a']:.0f} vs {r['med_b']:.0f}, p={r['p']:.3f}).")
            print("  => Significant but OPPOSITE direction - partial/qualified finding, "
                  "not classic injustice.")
    else:
        print(f"  No significant deprivation difference between misses and hits "
              f"(p={r['p']:.3f}). Misses are not concentrated by deprivation.")

print(f"\nNote: S1_ONLY n={counts['S1_ONLY']} is small, limiting power for Q1. "
      "Mann-Whitney treats hexes as independent; spatial clustering may inflate "
      "significance, so read effect sizes alongside p-values and treat results as "
      "indicative pending a spatially-aware test.")

Zone counts: {'BOTH': 550, 'S1_ONLY': 77, 'SEPA_ONLY': 2764, 'NEITHER': 810}
  BOTH (hits)            : 550
  S1_ONLY (SEPA misses)  : 77
  SEPA_ONLY (over-desig) : 2764
  NEITHER (correct null) : 810

Q1  SEPA MISSES (S1_ONLY) vs HITS (BOTH) - are overlooked-flood areas more vulnerable?
variable              med_miss   med_hit      diff   effect        p
simd_rank             3986.000  3986.000     0.000    -0.04    0.544 
age_dep_ratio            0.297     0.297     0.001    -0.01    0.946 
pop_density             32.524   450.873  -418.348     0.18    0.012*
dist_to_river_m        412.883   266.251   146.632    -0.32    0.000*
impervious_pct           1.078    13.462   -12.384     0.28    0.000*
mean_slope               8.174     5.272     2.903    -0.40    0.000*

Q2  SEPA OVER-DESIGNATIONS (SEPA_ONLY) vs HITS (BOTH) - do flagged-but-dry areas differ?
variable              med_over   med_hit      diff   effect        p
simd_rank             3924.528  3986.000   -61.472    -0.06    

In [ ]:
import numpy as np
import geopandas as gpd
from scipy.stats import mannwhitneyu

FULL_PATH = "hex_h3_res9_joined.gpkg"
SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"

# All 8 model predictors, grouped. VULN = social/vulnerability dimensions
# (the EJ evidence); HAZARD = physical flood controls (mechanism check).
# Uses raw (untransformed) columns for interpretable medians
VULN_VARS   = ["simd_rank", "age_dep_ratio", "pop_density",
               "greenspace_frac", "sewer_catchment_coverage"]
HAZARD_VARS = ["impervious_pct", "mean_slope", "dist_to_river_m"]
ALL_VARS    = VULN_VARS + HAZARD_VARS

# If any raw column is missing (only the sqrt_* version exists), fall back to it.
_RAW_TO_SQRT = {
    "pop_density": "sqrt_pop_density",
    "impervious_pct": "sqrt_impervious_pct",
    "mean_slope": "sqrt_mean_slope",
    "dist_to_river_m": "sqrt_dist_to_river_m",
}

# Scottish SIMD: rank 1 = MOST deprived -> LOWER simd_rank = MORE deprived.
SIMD_LOW_IS_DEPRIVED = True

ZONES = ["BOTH", "S1_ONLY", "SEPA_ONLY", "NEITHER"]
ZLABEL = {"BOTH":"hits", "S1_ONLY":"misses", "SEPA_ONLY":"over-desig.", "NEITHER":"correct null"}

# Load and clasify 
gdf = gpd.read_file(FULL_PATH)

# resolve each requested variable to a column that actually exists
def _resolve(cols):
    out = []
    for c in cols:
        if c in gdf.columns:
            out.append(c)
        elif _RAW_TO_SQRT.get(c) in gdf.columns:
            print(f"[note] '{c}' absent; using '{_RAW_TO_SQRT[c]}' instead.")
            out.append(_RAW_TO_SQRT[c])
        else:
            print(f"[warning] '{c}' not found and no fallback; skipping.")
    return out
VULN_VARS   = _resolve(VULN_VARS)
HAZARD_VARS = _resolve(HAZARD_VARS)
ALL_VARS    = VULN_VARS + HAZARD_VARS

sepa_f = gdf[SEPA_FRAC] > 0
s1_f   = gdf[S1_FRAC]   > 0

zone = np.full(len(gdf), "NEITHER", dtype=object)
zone[( s1_f &  sepa_f)] = "BOTH"
zone[( s1_f & ~sepa_f)] = "S1_ONLY"
zone[(~s1_f &  sepa_f)] = "SEPA_ONLY"
gdf["zone"] = zone
masks = {zn: gdf["zone"] == zn for zn in ZONES}

counts = {zn: int(masks[zn].sum()) for zn in ZONES}
N = len(gdf)
print("="*84)
print("ZONE COUNTS")
print("="*84)
for zn in ZONES:
    print(f"  {zn:<11} ({ZLABEL[zn]:<12}) : {counts[zn]:>6}  ({100*counts[zn]/N:5.1f}%)")
print(f"  {'TOTAL':<11} {'':<14} : {N:>6}  (100.0%)")

# FULL per-zone descriptive statistics (all four zones)
print("\n" + "="*84)
print("PER-ZONE DESCRIPTIVE STATISTICS  (median [IQR], mean)")
print("="*84)
for v in ALL_VARS:
    kind = "vuln" if v in VULN_VARS else "hazard"
    print(f"\n{v}  ({kind})")
    print(f"  {'zone':<12}{'n':>7}{'median':>12}{'mean':>12}{'Q1':>12}{'Q3':>12}")
    for zn in ZONES:
        s = gdf.loc[masks[zn], v].dropna()
        if len(s):
            print(f"  {ZLABEL[zn]:<12}{len(s):>7}{s.median():>12.3f}{s.mean():>12.3f}"
                  f"{s.quantile(.25):>12.3f}{s.quantile(.75):>12.3f}")
        else:
            print(f"  {ZLABEL[zn]:<12}{0:>7}{'--':>12}")

# Comparison helper function 
def compare(varname, mask_a, mask_b):
    a = gdf.loc[mask_a, varname].dropna().to_numpy()
    b = gdf.loc[mask_b, varname].dropna().to_numpy()
    if len(a) < 3 or len(b) < 3:
        return None
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    rbc = 1 - 2*U/(len(a)*len(b))                 # rank-biserial effect size
    return dict(na=len(a), nb=len(b), med_a=np.median(a), med_b=np.median(b),
                diff=np.median(a)-np.median(b), p=p, rbc=rbc)

def print_block(title, mask_a, label_a, mask_b, label_b):
    print("\n" + "="*84)
    print(title)
    print("="*84)
    print(f"{'variable':<20}{'med_'+label_a:>12}{'med_'+label_b:>12}{'diff':>10}{'effect':>9}{'p':>9}")
    for v in ALL_VARS:
        r = compare(v, mask_a, mask_b)
        if r:
            tag = "*" if r["p"] < 0.05 else " "
            print(f"{v:<20}{r['med_a']:>12.3f}{r['med_b']:>12.3f}{r['diff']:>10.3f}"
                  f"{r['rbc']:>9.2f}{r['p']:>9.3f}{tag}")

# Q1: misses vs hits (EJ core) 
print_block("Q1  MISSES (S1_ONLY) vs HITS (BOTH) - are overlooked-flood areas more vulnerable?",
            masks["S1_ONLY"], "miss", masks["BOTH"], "hit")

# Q2: over-designations vs hits 
print_block("Q2  OVER-DESIGNATIONS (SEPA_ONLY) vs HITS (BOTH)",
            masks["SEPA_ONLY"], "over", masks["BOTH"], "hit")

# Q3: (added, uses NEITHER) over-designations vs correct nulls 
# Do the areas SEPA wrongly flags differ from the areas it correctly leaves dry?
print_block("Q3  OVER-DESIGNATIONS (SEPA_ONLY) vs CORRECT NULLS (NEITHER) - what makes SEPA flag a dry hex?",
            masks["SEPA_ONLY"], "over", masks["NEITHER"], "null")

# Q4: (added, uses NEITHER) misses vs correct nulls
# Among the hexes SEPA did NOT flag, how do the ones that actually flooded
# (misses) differ from the ones that stayed dry (correct nulls)?
print_block("Q4  MISSES (S1_ONLY) vs CORRECT NULLS (NEITHER) - within SEPA's 'no-flag' set, what did it miss?",
            masks["S1_ONLY"], "miss", masks["NEITHER"], "null")

# EJ verdict on deprivation for the misses 
print("\n" + "="*84)
print("EJ VERDICT - are SEPA's MISSES concentrated in DEPRIVED areas?")
print("="*84)
r = compare("simd_rank", masks["S1_ONLY"], masks["BOTH"])
if r:
    if r["p"] < 0.05:
        miss_more_deprived = (r["diff"] < 0) if SIMD_LOW_IS_DEPRIVED else (r["diff"] > 0)
        if miss_more_deprived:
            print(f"  SEPA's misses fall in MORE deprived areas than its hits "
                  f"(median simd_rank {r['med_a']:.0f} vs {r['med_b']:.0f}, "
                  f"effect {r['rbc']:.2f}, p={r['p']:.3f}).")
            print("  => Representational injustice: overlooked flooding concentrated "
                  "on more deprived populations.")
        else:
            print(f"  SEPA's misses fall in LESS deprived areas than its hits "
                  f"(median {r['med_a']:.0f} vs {r['med_b']:.0f}, p={r['p']:.3f}).")
            print("  => Significant but OPPOSITE direction - partial/qualified finding.")
    else:
        print(f"  No significant deprivation difference between misses and hits "
              f"(p={r['p']:.3f}); misses are not concentrated by deprivation.")

print(f"\nNote: S1_ONLY n={counts['S1_ONLY']} is small, limiting power for Q1/Q4. "
      "Mann-Whitney treats hexes as independent; spatial clustering may inflate "
      "significance, so read effect sizes alongside p-values and treat as indicative.")

ZONE COUNTS
  BOTH        (hits        ) :    550  ( 13.1%)
  S1_ONLY     (misses      ) :     77  (  1.8%)
  SEPA_ONLY   (over-desig. ) :   2764  ( 65.8%)
  NEITHER     (correct null) :    810  ( 19.3%)
  TOTAL                      :   4201  (100.0%)

PER-ZONE DESCRIPTIVE STATISTICS  (median [IQR], mean)

simd_rank  (vuln)
  zone              n      median        mean          Q1          Q3
  hits            542    3986.000    3773.052    2617.000    5064.498
  misses           77    3986.000    3932.506    2896.513    5060.000
  over-desig.    2756    3924.528    4129.608    3873.000    4254.608
  correct null    798    3912.000    3929.763    3586.968    4238.000

age_dep_ratio  (vuln)
  zone              n      median        mean          Q1          Q3
  hits            424       0.297       0.308       0.239       0.366
  misses           48       0.297       0.309       0.230       0.378
  over-desig.     713       0.340       0.328       0.264       0.378
  correct null    115

# Disagreement-zone descriptive statistics: SEPA vs Sentinel-1

Every hex is classified into one of four clear categories by whether observed
(Sentinel-1) and modelled (SEPA) flooding agree:

    BOTH       flooded in both              (SEPA hit)
    S1_ONLY    flooded in S1 but not SEPA   (SEPA miss / omission)
    SEPA_ONLY  flooded in SEPA but not S1   (SEPA over-designation)
    NEITHER    dry in both                  (correct null)

For each of the 8 predictors, this reports the profile of each category
(count, median, mean, IQR). No pairwise tests,  just the four categories side
by side, so I can see how the misses, over-designations, hits and nulls differ.


In [ ]:
import numpy as np
import geopandas as gpd

FULL_PATH = "hex_h3_res9_joined.gpkg"
SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"

VULN_VARS   = ["simd_rank", "age_dep_ratio", "pop_density",
               "greenspace_frac", "sewer_catchment_coverage"]
HAZARD_VARS = ["impervious_pct", "mean_slope", "dist_to_river_m"]
ALL_VARS    = VULN_VARS + HAZARD_VARS

_RAW_TO_SQRT = {
    "pop_density": "sqrt_pop_density",
    "impervious_pct": "sqrt_impervious_pct",
    "mean_slope": "sqrt_mean_slope",
    "dist_to_river_m": "sqrt_dist_to_river_m",
}

ZONES  = ["BOTH", "S1_ONLY", "SEPA_ONLY", "NEITHER"]
ZLABEL = {"BOTH":"hits", "S1_ONLY":"misses", "SEPA_ONLY":"over-desig.", "NEITHER":"correct null"}

# Load and resolve columns 
gdf = gpd.read_file(FULL_PATH)

def _resolve(cols):
    out = []
    for c in cols:
        if c in gdf.columns:
            out.append(c)
        elif _RAW_TO_SQRT.get(c) in gdf.columns:
            print(f"[note] '{c}' absent; using '{_RAW_TO_SQRT[c]}'.")
            out.append(_RAW_TO_SQRT[c])
        else:
            print(f"[warning] '{c}' not found; skipping.")
    return out
VULN_VARS   = _resolve(VULN_VARS)
HAZARD_VARS = _resolve(HAZARD_VARS)
ALL_VARS    = VULN_VARS + HAZARD_VARS

# Classify 
sepa_f = gdf[SEPA_FRAC] > 0
s1_f   = gdf[S1_FRAC]   > 0
zone = np.full(len(gdf), "NEITHER", dtype=object)
zone[( s1_f &  sepa_f)] = "BOTH"
zone[( s1_f & ~sepa_f)] = "S1_ONLY"
zone[(~s1_f &  sepa_f)] = "SEPA_ONLY"
gdf["zone"] = zone
masks = {zn: gdf["zone"] == zn for zn in ZONES}

N = len(gdf)
counts = {zn: int(masks[zn].sum()) for zn in ZONES}

print("="*76)
print("ZONE COUNTS")
print("="*76)
for zn in ZONES:
    print(f"  {zn:<11} ({ZLABEL[zn]:<12}) : {counts[zn]:>6}  ({100*counts[zn]/N:5.1f}%)")
print(f"  {'TOTAL':<11} {'':<14} : {N:>6}  (100.0%)")

# Per-variable, four categories side by side 
print("\n" + "="*76)
print("PREDICTOR PROFILE BY ZONE  (n, median, mean, Q1, Q3)")
print("="*76)
for v in ALL_VARS:
    kind = "vulnerability" if v in VULN_VARS else "hazard"
    print(f"\n{v}  [{kind}]")
    print(f"  {'zone':<14}{'n':>7}{'median':>12}{'mean':>12}{'Q1':>12}{'Q3':>12}")
    print(f"  {'-'*12:<14}{'-'*6:>7}{'-'*11:>12}{'-'*11:>12}{'-'*11:>12}{'-'*11:>12}")
    for zn in ZONES:
        s = gdf.loc[masks[zn], v].dropna()
        if len(s):
            print(f"  {ZLABEL[zn]:<14}{len(s):>7}{s.median():>12.3f}{s.mean():>12.3f}"
                  f"{s.quantile(.25):>12.3f}{s.quantile(.75):>12.3f}")
        else:
            print(f"  {ZLABEL[zn]:<14}{0:>7}{'--':>12}")

# Tidy table 
import pandas as pd
rows = []
for v in ALL_VARS:
    for zn in ZONES:
        s = gdf.loc[masks[zn], v].dropna()
        rows.append({
            "variable": v,
            "group": "vulnerability" if v in VULN_VARS else "hazard",
            "zone": ZLABEL[zn],
            "n": len(s),
            "median": round(s.median(), 3) if len(s) else np.nan,
            "mean": round(s.mean(), 3) if len(s) else np.nan,
            "Q1": round(s.quantile(.25), 3) if len(s) else np.nan,
            "Q3": round(s.quantile(.75), 3) if len(s) else np.nan,
        })
tab = pd.DataFrame(rows)
tab.to_csv("disagreement_zone_profile.csv", index=False)
print("\nSaved tidy table: disagreement_zone_profile.csv")

ZONE COUNTS
  BOTH        (hits        ) :    550  ( 13.1%)
  S1_ONLY     (misses      ) :     77  (  1.8%)
  SEPA_ONLY   (over-desig. ) :   2764  ( 65.8%)
  NEITHER     (correct null) :    810  ( 19.3%)
  TOTAL                      :   4201  (100.0%)

PREDICTOR PROFILE BY ZONE  (n, median, mean, Q1, Q3)

simd_rank  [vulnerability]
  zone                n      median        mean          Q1          Q3
  ------------   ------ ----------- ----------- ----------- -----------
  hits              542    3986.000    3773.052    2617.000    5064.498
  misses             77    3986.000    3932.506    2896.513    5060.000
  over-desig.      2756    3924.528    4129.608    3873.000    4254.608
  correct null      798    3912.000    3929.763    3586.968    4238.000

age_dep_ratio  [vulnerability]
  zone                n      median        mean          Q1          Q3
  ------------   ------ ----------- ----------- ----------- -----------
  hits              424       0.297       0.308       0.23

In [ ]:
import numpy as np
import geopandas as gpd


FULL_PATH = "hex_h3_res9_joined.gpkg"
SEPA_FRAC = "sepa_flood_frac"
S1_FRAC   = "s1_flood_frac"

VULN_VARS = ["simd_rank", "age_dep_ratio", "pop_density"]     # social/vulnerability
HAZARD_VARS = ["dist_to_river_m", "impervious_pct", "mean_slope"]  # mechanism check


SIMD_LOW_IS_DEPRIVED = True

# Load and classify 
gdf = gpd.read_file(FULL_PATH)
sepa_f = gdf[SEPA_FRAC] > 0
s1_f   = gdf[S1_FRAC]   > 0

zone = np.full(len(gdf), "NEITHER", dtype=object)
zone[( s1_f &  sepa_f)] = "BOTH"
zone[( s1_f & ~sepa_f)] = "S1_ONLY"
zone[(~s1_f &  sepa_f)] = "SEPA_ONLY"
gdf["zone"] = zone

counts = {z: int((zone == z).sum()) for z in ["BOTH","S1_ONLY","SEPA_ONLY","NEITHER"]}
print("Zone counts:", counts)
print(f"  BOTH (hits)            : {counts['BOTH']}")
print(f"  S1_ONLY (SEPA misses)  : {counts['S1_ONLY']}")
print(f"  SEPA_ONLY (over-desig) : {counts['SEPA_ONLY']}")
print(f"  NEITHER (correct null) : {counts['NEITHER']}")

def summarise(varname, mask):
    v = gdf.loc[mask, varname].dropna()
    return len(v), v.median(), v.mean(), v.quantile(.25), v.quantile(.75)

both_m  = gdf["zone"]=="BOTH"
s1_m    = gdf["zone"]=="S1_ONLY"
sepa_m  = gdf["zone"]=="SEPA_ONLY"
neither_m = gdf["zone"]=="NEITHER"

# Per-zone profile: each category on its own 
ZONE_ROWS = [
    ("BOTH (hits)",           both_m),
    ("S1_ONLY (misses)",      s1_m),
    ("SEPA_ONLY (over-desig)",sepa_m),
    ("NEITHER (correct null)",neither_m),
]

print("\n" + "="*84)
print("PER-ZONE PROFILE  (n, median, mean, Q1, Q3) - each zone independently")
print("="*84)
for v in VULN_VARS + HAZARD_VARS:
    kind = "vulnerability" if v in VULN_VARS else "hazard"
    print(f"\n{v}  [{kind}]")
    print(f"  {'zone':<26}{'n':>7}{'median':>11}{'mean':>11}{'Q1':>11}{'Q3':>11}")
    for label, m in ZONE_ROWS:
        nn, med, mean, q1, q3 = summarise(v, m)
        print(f"  {label:<26}{nn:>7}{med:>11.3f}{mean:>11.3f}{q1:>11.3f}{q3:>11.3f}")

Zone counts: {'BOTH': 550, 'S1_ONLY': 77, 'SEPA_ONLY': 2764, 'NEITHER': 810}
  BOTH (hits)            : 550
  S1_ONLY (SEPA misses)  : 77
  SEPA_ONLY (over-desig) : 2764
  NEITHER (correct null) : 810

PER-ZONE PROFILE  (n, median, mean, Q1, Q3) - each zone independently

simd_rank  [vulnerability]
  zone                            n     median       mean         Q1         Q3
  BOTH (hits)                   542   3986.000   3773.052   2617.000   5064.498
  S1_ONLY (misses)               77   3986.000   3932.506   2896.513   5060.000
  SEPA_ONLY (over-desig)       2756   3924.528   4129.608   3873.000   4254.608
  NEITHER (correct null)        798   3912.000   3929.763   3586.968   4238.000

age_dep_ratio  [vulnerability]
  zone                            n     median       mean         Q1         Q3
  BOTH (hits)                   424      0.297      0.308      0.239      0.366
  S1_ONLY (misses)               48      0.297      0.309      0.230      0.378
  SEPA_ONLY (over-desig)    